# Gibbs States: Thermal Weight Solver

This notebook constructs a finite-temperature Gibbs state `rho = exp(-beta H) / Z` for a small Hamiltonian using a bounded exponential polynomial.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsvt.hamiltonians import ising_hamiltonian
from qsvt.spectral import (
    apply_function_to_hermitian,
    apply_polynomial_to_hermitian,
    eigh_hermitian,
)
from qsvt.templates import exponential_approximation_polynomial

H = ising_hamiltonian(2, coupling=1.0, transverse_field=0.6)
energies, _ = eigh_hermitian(H)
beta_phys = 1.7

center = 0.5 * (energies[0] + energies[-1])
half_width = 0.5 * (energies[-1] - energies[0])
A = (center * np.eye(H.shape[0]) - H) / half_width
poly_beta = beta_phys * half_width
prefactor = np.exp(-beta_phys * center + poly_beta)

coeffs = exponential_approximation_polynomial(degree=16, beta=poly_beta)
thermal_poly = prefactor * apply_polynomial_to_hermitian(A, coeffs)
rho_poly = thermal_poly / np.trace(thermal_poly)

thermal_exact = apply_function_to_hermitian(H, lambda e: np.exp(-beta_phys * e))
rho_exact = thermal_exact / np.trace(thermal_exact)
rho_error = np.linalg.norm(rho_poly - rho_exact)
rho_error

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(
    np.arange(len(energies)) - 0.18, np.linalg.eigvalsh(rho_exact), 0.36, label="exact"
)
plt.bar(
    np.arange(len(energies)) + 0.18,
    np.linalg.eigvalsh(rho_poly),
    0.36,
    label="polynomial",
)
plt.xlabel("thermal eigenstate index")
plt.ylabel("Gibbs probability")
plt.title(f"Gibbs state approximation, density-matrix error={rho_error:.2e}")
plt.legend()
plt.show()